# BERT 기반 한국어 영화 리뷰 감성 분류 - PyTorch

1. `transformers`, `torch` 등 필요한 패키지 설치 및 불러오기
2. 네이버 영화 리뷰 데이터셋(NSMC) 다운로드 및 읽기
3. 결측치 제거와 데이터 확인
4. KLUE BERT 토크나이저를 활용한 토큰화, 정수 인코딩, 디코딩 확인
5. 최대 길이 128 기준으로 `input_ids`, `attention_mask`, `token_type_ids` 생성
6. `torch.utils.data.Dataset`과 `DataLoader` 구성
7. `BertModel` 위에 이진 분류기를 추가한 PyTorch 모델 설계
8. 학습, 평가, 예측 함수 작성
9. 새로운 영화 리뷰 문장에 대한 긍정/부정 예측 수행

> 실행 시간을 줄이기 위해 기본값은 일부 샘플만 사용하도록 설정되어 있습니다. 전체 데이터를 학습하려면 `USE_SMALL_SAMPLE = False`로 변경하면 됩니다.


## 패키지 설치, 데이터 다운로드, 결측치 제거

 BERT 영화 리뷰 분류 실습의 시작 단계로 `transformers` 설치, 필요한 패키지 import, 훈련용·평가용 데이터 파일 다운로드, 데이터 읽기, 상위 5개 데이터 확인, Null 값 제거 과정을 설명합니다.

데이터 개수 출력, 상위 데이터 확인, `dropna()`를 이용한 결측치 제거 결과를 보여줍니다. 이 코드에서는 같은 흐름을 PyTorch 학습에 필요한 형태로 준비합니다.


In [1]:
# Colab 또는 로컬 환경에서 필요한 패키지를 설치합니다.
# - transformers: Hugging Face의 BERT 모델과 토크나이저를 사용하기 위한 패키지입니다.
# - torch: PyTorch 딥러닝 프레임워크입니다.
# - pandas: 표 형태의 데이터셋을 읽고 전처리하기 위한 패키지입니다.
# - tqdm: 반복문의 진행률을 보기 좋게 출력하기 위한 패키지입니다.
%pip install -q transformers torch pandas tqdm scikit-learn


In [2]:
# 운영체제 기능을 사용하기 위한 표준 라이브러리입니다.
import os

# URL에서 파일을 다운로드하기 위한 표준 라이브러리입니다.
import urllib.request

# 난수 고정을 위해 사용하는 표준 라이브러리입니다.
import random

# 수치 계산과 배열 처리를 위해 사용하는 라이브러리입니다.
import numpy as np

# 표 형태의 데이터를 읽고 처리하기 위해 사용하는 라이브러리입니다.
import pandas as pd

# 반복 작업의 진행률을 출력하기 위해 사용하는 라이브러리입니다.
from tqdm.auto import tqdm

# PyTorch의 핵심 기능을 사용하기 위한 라이브러리입니다.
import torch

# PyTorch에서 신경망 계층과 손실함수를 만들기 위한 모듈입니다.
import torch.nn as nn

# PyTorch에서 최적화 알고리즘을 사용하기 위한 모듈입니다.
import torch.optim as optim

# PyTorch에서 데이터셋과 미니배치 로더를 만들기 위한 클래스입니다.
from torch.utils.data import Dataset, DataLoader

# 학습 데이터와 검증 데이터를 나누기 위한 scikit-learn 함수입니다.
from sklearn.model_selection import train_test_split

# Hugging Face에서 BERT 토크나이저와 BERT 본체 모델을 불러오기 위한 클래스입니다.
from transformers import BertTokenizer, BertModel


In [3]:
# 실험을 재현하기 위해 사용할 난수 시드 값을 지정합니다.
SEED = 42

# 파이썬 random 모듈의 난수를 고정합니다.
random.seed(SEED)

# NumPy 난수를 고정합니다.
np.random.seed(SEED)

# PyTorch CPU 연산의 난수를 고정합니다.
torch.manual_seed(SEED)

# CUDA GPU를 사용할 수 있는 경우 GPU 연산의 난수도 고정합니다.
torch.cuda.manual_seed_all(SEED)

# 현재 실행 환경에서 CUDA GPU를 사용할 수 있으면 'cuda', 아니면 'cpu'를 선택합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 선택된 장치를 출력하여 학습이 CPU에서 실행되는지 GPU에서 실행되는지 확인합니다.
print("사용 장치:", device)


사용 장치: cuda


In [4]:
# 네이버 영화 리뷰 훈련 데이터가 저장될 파일 이름을 지정합니다.
train_file = "ratings_train.txt"

# 네이버 영화 리뷰 테스트 데이터가 저장될 파일 이름을 지정합니다.
test_file = "ratings_test.txt"

# 훈련 데이터 다운로드 주소를 지정합니다.
train_url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"

# 테스트 데이터 다운로드 주소를 지정합니다.
test_url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt"

# 훈련 데이터 파일이 현재 폴더에 없을 때만 다운로드합니다.
if not os.path.exists(train_file):
    # URL에서 훈련 데이터 파일을 다운로드하여 ratings_train.txt로 저장합니다.
    urllib.request.urlretrieve(train_url, filename=train_file)

# 테스트 데이터 파일이 현재 폴더에 없을 때만 다운로드합니다.
if not os.path.exists(test_file):
    # URL에서 테스트 데이터 파일을 다운로드하여 ratings_test.txt로 저장합니다.
    urllib.request.urlretrieve(test_url, filename=test_file)

# 다운로드가 완료되었음을 출력합니다.
print("데이터 다운로드 확인 완료")


데이터 다운로드 확인 완료


In [5]:
# 탭으로 구분된 훈련용 텍스트 파일을 pandas DataFrame으로 읽습니다.
train_data = pd.read_table(train_file)

# 탭으로 구분된 테스트용 텍스트 파일을 pandas DataFrame으로 읽습니다.
test_data = pd.read_table(test_file)

# 훈련용 리뷰 개수를 출력합니다.
print("훈련용 리뷰 개수:", len(train_data))

# 테스트용 리뷰 개수를 출력합니다.
print("테스트용 리뷰 개수:", len(test_data))

# 훈련 데이터의 상위 5개 행을 확인합니다.
display(train_data.head())

# 테스트 데이터의 상위 5개 행을 확인합니다.
display(test_data.head())


훈련용 리뷰 개수: 150000
테스트용 리뷰 개수: 50000


,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


,id,document,label
0,6270596,굳 ㅋ,1
1,9274899,GDNTOPCLASSINTHECLUB,0
2,8544678,뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아,0
3,6825595,지루하지는 않은데 완전 막장임... 돈주고 보기에는....,0
4,6723715,3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??,0


In [6]:
# document 또는 label에 결측치가 있는 훈련 데이터 행을 제거합니다.
train_data = train_data.dropna(how="any")

# 결측치 제거 후 인덱스를 0부터 다시 정리합니다.
train_data = train_data.reset_index(drop=True)

# document 또는 label에 결측치가 있는 테스트 데이터 행을 제거합니다.
test_data = test_data.dropna(how="any")

# 결측치 제거 후 인덱스를 0부터 다시 정리합니다.
test_data = test_data.reset_index(drop=True)

# 훈련 데이터에 결측치가 남아 있는지 True 또는 False로 확인합니다.
print("훈련 데이터 결측치 존재 여부:", train_data.isnull().values.any())

# 테스트 데이터에 결측치가 남아 있는지 True 또는 False로 확인합니다.
print("테스트 데이터 결측치 존재 여부:", test_data.isnull().values.any())

# 결측치 제거 후 훈련 데이터 개수를 출력합니다.
print("결측치 제거 후 훈련용 리뷰 개수:", len(train_data))

# 결측치 제거 후 테스트 데이터 개수를 출력합니다.
print("결측치 제거 후 테스트용 리뷰 개수:", len(test_data))


훈련 데이터 결측치 존재 여부: False
테스트 데이터 결측치 존재 여부: False
결측치 제거 후 훈련용 리뷰 개수: 149995
결측치 제거 후 테스트용 리뷰 개수: 49997


##  KLUE BERT 토크나이저, WordPiece, CLS/SEP/PAD 토큰 확인

기존에 학습된 BERT 모델을 활용하고, 문장을 정수로 인코딩하며, 토큰 단위로 분리하는 과정을 설명합니다. BERT는 WordPiece 방식을 사용하기 때문에 사전에 없는 단어를 더 작은 조각으로 나누고, 중간 조각에는 `##` 표시가 붙을 수 있습니다.

`tokenizer.encode()`, `tokenizer.tokenize()`, `tokenizer.decode()` 결과와 `[CLS]`, `[SEP]`, `[PAD]` 특수 토큰을 확인하는 출력 예시를 보여줍니다.


In [7]:
# 사용할 한국어 BERT 모델 이름을 지정합니다.
MODEL_NAME = "klue/bert-base"

# KLUE BERT 모델에 맞는 토크나이저를 Hugging Face 저장소에서 불러옵니다.
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

# 토크나이저가 정상적으로 불러와졌는지 모델 이름을 출력합니다.
print("사용 토크나이저:", MODEL_NAME)

# BERT 문장 시작 토큰과 해당 정수 ID를 출력합니다.
print("CLS 토큰:", tokenizer.cls_token, tokenizer.cls_token_id)

# BERT 문장 종료 토큰과 해당 정수 ID를 출력합니다.
print("SEP 토큰:", tokenizer.sep_token, tokenizer.sep_token_id)

# BERT 패딩 토큰과 해당 정수 ID를 출력합니다.
print("PAD 토큰:", tokenizer.pad_token, tokenizer.pad_token_id)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

사용 토크나이저: klue/bert-base
CLS 토큰: [CLS] 2
SEP 토큰: [SEP] 3
PAD 토큰: [PAD] 0


In [8]:
# 토큰화와 인코딩을 확인할 예시 문장을 지정합니다.
sample_sentence = "보는내내 그대로 들어맞는 예측 카리스마 없는 악역"

# 예시 문장을 BERT 토큰 단위로 분리합니다.
tokens = tokenizer.tokenize(sample_sentence)

# 예시 문장을 BERT 정수 ID 시퀀스로 변환합니다.
encoded_ids = tokenizer.encode(sample_sentence)

# 토큰화 결과를 출력합니다.
print("토큰화 결과:", tokens)

# 정수 인코딩 결과를 출력합니다.
print("정수 인코딩 결과:", encoded_ids)

# 정수 ID 시퀀스를 다시 문자열로 복원하여 출력합니다.
print("디코딩 결과:", tokenizer.decode(encoded_ids))


토큰화 결과: ['보', '##는', '##내', '##내', '그대로', '들어맞', '##는', '예측', '카리스마', '없', '##는', '악역']
정수 인코딩 결과: [2, 1160, 2259, 2369, 2369, 4311, 20657, 2259, 5501, 13132, 1415, 2259, 23713, 3]
디코딩 결과: [CLS] 보는내내 그대로 들어맞는 예측 카리스마 없는 악역 [SEP]


In [9]:
# 정수 ID 하나하나가 어떤 토큰으로 복원되는지 확인합니다.
for token_id in encoded_ids:
    # 현재 정수 ID를 토큰 문자열로 디코딩합니다.
    decoded_token = tokenizer.decode(token_id)

    # 정수 ID와 해당 토큰을 함께 출력합니다.
    print(token_id, "->", decoded_token)


2 -> [CLS]
1160 -> 보
2259 -> ##는
2369 -> ##내
2369 -> ##내
4311 -> 그대로
20657 -> 들어맞
2259 -> ##는
5501 -> 예측
13132 -> 카리스마
1415 -> 없
2259 -> ##는
23713 -> 악역
3 -> [SEP]


In [10]:
# WordPiece 분리를 확인할 두 번째 예시 문장을 지정합니다.
oov_sentence = "전율을 일으키는 영화. 다시 보고싶은 영화"

# 두 번째 예시 문장의 토큰화 결과를 출력합니다.
print("토큰화 결과:", tokenizer.tokenize(oov_sentence))

# 두 번째 예시 문장의 정수 인코딩 결과를 출력합니다.
print("정수 인코딩 결과:", tokenizer.encode(oov_sentence))

# 영어 문장도 같은 토크나이저 규칙으로 분리되는지 확인합니다.
for token_id in tokenizer.encode("happy birthday~!"):
    # 영어 예시 문장의 각 정수 ID를 토큰으로 복원합니다.
    print(token_id, "->", tokenizer.decode(token_id))


토큰화 결과: ['전', '##율', '##을', '일으키', '##는', '영화', '.', '다시', '보고', '##싶', '##은', '영화']
정수 인코딩 결과: [2, 1537, 2534, 2069, 6572, 2259, 3771, 18, 3690, 4530, 2585, 2073, 3771, 3]
2 -> [CLS]
13866 -> ha
11177 -> ##pp
2076 -> ##y
69 -> b
6492 -> ##ir
7088 -> ##th
27697 -> ##day
97 -> ~
5 -> !
3 -> [SEP]


## 최대 길이 128, 패딩, 어텐션 마스크, Dataset 변환

문장의 최대 길이를 128로 설정하고, 짧은 문장의 남는 위치를 0으로 채우는 패딩을 설명합니다. 또한 실제 단어 위치와 패딩 위치를 구분하기 위해 `attention_mask`를 사용합니다.

`input_ids`, `attention_mask`, `token_type_ids`를 생성하고, 훈련용·평가용 데이터 전체를 같은 방식으로 변환한 뒤 샘플 하나를 출력하는 과정을 보여줍니다. PyTorch에서는 이 데이터를 `Dataset`과 `DataLoader`로 구성하여 미니배치 학습에 사용합니다.


In [11]:
# BERT에 입력할 문장의 최대 토큰 길이를 128로 설정합니다.
MAX_SEQ_LEN = 128

# 토크나이저가 실제 학습 입력을 만드는 방식을 하나의 예시 문장으로 확인합니다.
encoded_sample = tokenizer(
    oov_sentence,                 # 인코딩할 문장을 입력합니다.
    max_length=MAX_SEQ_LEN,        # 모든 문장을 최대 128 토큰 길이에 맞춥니다.
    padding="max_length",         # 최대 길이보다 짧은 문장은 PAD 토큰으로 채웁니다.
    truncation=True,               # 최대 길이보다 긴 문장은 뒤쪽을 잘라냅니다.
    return_tensors=None            # 파이썬 리스트 형태로 결과를 받습니다.
)

# 단어와 특수 토큰이 정수 ID로 변환된 결과를 출력합니다.
print("input_ids:", encoded_sample["input_ids"])

# 실제 토큰 위치는 1, 패딩 위치는 0으로 표시한 마스크를 출력합니다.
print("attention_mask:", encoded_sample["attention_mask"])

# 한 문장 분류 문제이므로 모든 위치가 0인 세그먼트 ID를 출력합니다.
print("token_type_ids:", encoded_sample["token_type_ids"])

# input_ids 길이가 128인지 확인합니다.
print("input_ids 길이:", len(encoded_sample["input_ids"]))

# 정수 ID를 다시 문자열로 복원하여 확인합니다.
print("복원 문장:", tokenizer.decode(encoded_sample["input_ids"]))


input_ids: [2, 1537, 2534, 2069, 6572, 2259, 3771, 18, 3690, 4530, 2585, 2073, 3771, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
token_type_ids: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [12]:
# PyTorch 학습용 데이터셋 클래스를 정의합니다.
class NSMCDataset(Dataset):
    # 데이터셋 객체가 생성될 때 실행되는 초기화 메서드입니다.
    def __init__(self, texts, labels, tokenizer, max_seq_len):
        # 리뷰 문장 목록을 리스트 형태로 저장합니다.
        self.texts = list(texts)

        # 정답 라벨 목록을 정수 리스트 형태로 저장합니다.
        self.labels = list(labels)

        # BERT 입력 변환에 사용할 토크나이저를 저장합니다.
        self.tokenizer = tokenizer

        # 모든 문장을 맞출 최대 토큰 길이를 저장합니다.
        self.max_seq_len = max_seq_len

    # 데이터셋에 포함된 전체 샘플 개수를 반환합니다.
    def __len__(self):
        # 리뷰 문장 개수를 반환합니다.
        return len(self.texts)

    # 특정 인덱스의 샘플 하나를 반환합니다.
    def __getitem__(self, idx):
        # idx 위치의 리뷰 문장을 문자열로 가져옵니다.
        text = str(self.texts[idx])

        # idx 위치의 라벨을 정수로 가져옵니다.
        label = int(self.labels[idx])

        # 리뷰 문장을 BERT 입력 형식으로 변환합니다.
        encoding = self.tokenizer(
            text,                         # 변환할 리뷰 문장을 입력합니다.
            max_length=self.max_seq_len,   # 최대 길이를 128로 제한합니다.
            padding="max_length",         # 짧은 문장은 PAD 토큰으로 채웁니다.
            truncation=True,               # 긴 문장은 최대 길이에 맞게 자릅니다.
            return_tensors="pt"            # 결과를 PyTorch 텐서 형태로 반환합니다.
        )

        # 모델에 입력할 input_ids, attention_mask, token_type_ids와 라벨을 딕셔너리로 반환합니다.
        return {
            # squeeze(0)는 [1, 128] 형태를 [128] 형태로 바꾸어 DataLoader가 배치 차원을 만들게 합니다.
            "input_ids": encoding["input_ids"].squeeze(0),

            # attention_mask도 [128] 형태로 반환합니다.
            "attention_mask": encoding["attention_mask"].squeeze(0),

            # token_type_ids도 [128] 형태로 반환합니다.
            "token_type_ids": encoding["token_type_ids"].squeeze(0),

            # 이진 분류 라벨은 BCEWithLogitsLoss에 맞게 float 텐서로 반환합니다.
            "label": torch.tensor(label, dtype=torch.float)
        }


In [13]:
# 실습 실행 시간을 줄이기 위해 일부 샘플만 사용할지 결정합니다.
USE_SMALL_SAMPLE = True

# 빠른 실습에 사용할 훈련 샘플 수를 지정합니다.
TRAIN_SAMPLE_SIZE = 3000

# 빠른 실습에 사용할 테스트 샘플 수를 지정합니다.
TEST_SAMPLE_SIZE = 1000

# 일부 샘플만 사용할 경우 데이터프레임에서 앞쪽 일부만 복사합니다.
if USE_SMALL_SAMPLE:
    # 훈련 데이터에서 지정한 개수만큼만 사용합니다.
    train_work = train_data.sample(n=min(TRAIN_SAMPLE_SIZE, len(train_data)), random_state=SEED).reset_index(drop=True)

    # 테스트 데이터에서 지정한 개수만큼만 사용합니다.
    test_work = test_data.sample(n=min(TEST_SAMPLE_SIZE, len(test_data)), random_state=SEED).reset_index(drop=True)
else:
    # 전체 훈련 데이터를 사용합니다.
    train_work = train_data.reset_index(drop=True)

    # 전체 테스트 데이터를 사용합니다.
    test_work = test_data.reset_index(drop=True)

# 훈련 데이터를 실제 훈련용과 검증용으로 나눕니다.
train_df, valid_df = train_test_split(
    train_work,                 # 분할할 훈련 데이터프레임입니다.
    test_size=0.2,              # 전체 중 20%를 검증용으로 사용합니다.
    random_state=SEED,          # 같은 결과가 나오도록 난수를 고정합니다.
    stratify=train_work["label"] # 긍정/부정 비율이 유지되도록 라벨 기준 층화 분할을 합니다.
)

# 분할된 데이터의 인덱스를 다시 정리합니다.
train_df = train_df.reset_index(drop=True)

# 분할된 검증 데이터의 인덱스를 다시 정리합니다.
valid_df = valid_df.reset_index(drop=True)

# 각 데이터 개수를 출력합니다.
print("실제 학습 데이터 개수:", len(train_df))
print("검증 데이터 개수:", len(valid_df))
print("테스트 데이터 개수:", len(test_work))


실제 학습 데이터 개수: 2400
검증 데이터 개수: 600
테스트 데이터 개수: 1000


In [14]:
# 훈련용 Dataset 객체를 생성합니다.
train_dataset = NSMCDataset(train_df["document"], train_df["label"], tokenizer, MAX_SEQ_LEN)

# 검증용 Dataset 객체를 생성합니다.
valid_dataset = NSMCDataset(valid_df["document"], valid_df["label"], tokenizer, MAX_SEQ_LEN)

# 테스트용 Dataset 객체를 생성합니다.
test_dataset = NSMCDataset(test_work["document"], test_work["label"], tokenizer, MAX_SEQ_LEN)

# 학습용 미니배치 크기를 지정합니다.
BATCH_SIZE = 16

# 훈련용 DataLoader를 생성합니다.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# 검증용 DataLoader를 생성합니다.
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 테스트용 DataLoader를 생성합니다.
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 첫 번째 훈련 샘플을 가져옵니다.
sample_item = train_dataset[0]

# 첫 번째 샘플의 input_ids를 출력합니다.
print("단어에 대한 정수 인코딩:", sample_item["input_ids"])

# 첫 번째 샘플의 attention_mask를 출력합니다.
print("어텐션 마스크:", sample_item["attention_mask"])

# 첫 번째 샘플의 token_type_ids를 출력합니다.
print("세그먼트 인코딩:", sample_item["token_type_ids"])

# 첫 번째 샘플의 input_ids 길이를 출력합니다.
print("각 인코딩의 길이:", len(sample_item["input_ids"]))

# 첫 번째 샘플의 정수 인코딩을 다시 문장으로 복원하여 출력합니다.
print("정수 인코딩 복원:", tokenizer.decode(sample_item["input_ids"]))

# 첫 번째 샘플의 정답 라벨을 출력합니다.
print("레이블:", sample_item["label"].item())


단어에 대한 정수 인코딩: tensor([    2, 20609,  2154,   772,  2088,  5429,  2179,  3771,  2116,  1039,
         2073,  2147,    16,  1504,  3771,  2259,  4254, 20609,  2052,    27,
         2532,  2772,  2170,  1378,  2496, 31369,  3944,  5825,  2470,  3771,
         2507,  2062,    18,     3,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,

## KLUE BERT 불러오기와 Many-to-One 분류 모델 설계

 한국어 BERT 모델 `klue/bert-base`를 불러오고, BERT 출력의 0번과 1번 결과를 확인하는 흐름을 설명합니다. 일반적으로 BERT의 마지막 은닉 상태는 `[batch, seq_len, hidden_size]` 형태이고, 문장 전체 분류에는 `[CLS]` 위치의 표현 또는 pooler 출력을 사용합니다.

기존 BERT 모델 위에 분류층을 추가하는 예시를 보여줍니다. TensorFlow 코드에서는 sigmoid 출력층을 사용하지만, PyTorch에서는 수치 안정성을 위해 모델이 확률이 아닌 로짓(logit)을 출력하고, 손실함수로 `BCEWithLogitsLoss`를 사용합니다.


In [15]:
# BERT 기반 이진 감성 분류 모델 클래스를 정의합니다.
class BertForBinaryClassification(nn.Module):
    # 모델 객체가 생성될 때 실행되는 초기화 메서드입니다.
    def __init__(self, model_name):
        # 부모 클래스인 nn.Module의 초기화 메서드를 실행합니다.
        super().__init__()

        # 사전 학습된 KLUE BERT 본체 모델을 불러옵니다.
        self.bert = BertModel.from_pretrained(model_name)

        # 과적합을 줄이기 위해 분류층 앞에 Dropout을 적용합니다.
        self.dropout = nn.Dropout(p=0.1)

        # BERT의 hidden_size 값을 가져옵니다. klue/bert-base는 일반적으로 768입니다.
        hidden_size = self.bert.config.hidden_size

        # hidden_size 차원의 CLS 표현을 1개의 로짓으로 바꾸는 선형 분류층입니다.
        self.classifier = nn.Linear(hidden_size, 1)

    # 모델의 순전파 계산을 정의합니다.
    def forward(self, input_ids, attention_mask, token_type_ids):
        # BERT 본체에 토큰 ID, 어텐션 마스크, 세그먼트 ID를 입력합니다.
        outputs = self.bert(
            input_ids=input_ids,                 # 문장을 정수 ID로 표현한 입력입니다.
            attention_mask=attention_mask,       # 실제 토큰과 패딩을 구분하는 마스크입니다.
            token_type_ids=token_type_ids        # 한 문장 입력에서는 대부분 0으로 구성됩니다.
        )

        # pooler_output은 CLS 토큰을 기반으로 문장 전체 의미를 요약한 벡터입니다.
        pooled_output = outputs.pooler_output

        # Dropout을 적용하여 일부 뉴런을 무작위로 비활성화합니다.
        dropped_output = self.dropout(pooled_output)

        # 선형 분류층을 통과시켜 긍정 클래스에 대한 로짓을 계산합니다.
        logits = self.classifier(dropped_output)

        # [batch_size, 1] 형태를 [batch_size] 형태로 바꿔 손실 계산을 쉽게 합니다.
        return logits.squeeze(-1)


In [18]:
# BERT 이진 분류 모델 객체를 생성합니다.
model = BertForBinaryClassification(MODEL_NAME)

# 모델을 CPU 또는 GPU 장치로 이동합니다.
model = model.to(device)

# 모델 구조를 출력하여 BERT와 분류층이 포함되었는지 확인합니다.
print(model)


config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForBinaryClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, e

In [19]:
# DataLoader에서 첫 번째 미니배치를 가져옵니다.
batch = next(iter(train_loader))

# input_ids 텐서를 학습 장치로 이동합니다.
input_ids = batch["input_ids"].to(device)

# attention_mask 텐서를 학습 장치로 이동합니다.
attention_mask = batch["attention_mask"].to(device)

# token_type_ids 텐서를 학습 장치로 이동합니다.
token_type_ids = batch["token_type_ids"].to(device)

# 기울기 계산을 하지 않는 상태에서 출력 형태만 확인합니다.
with torch.no_grad():
    # 모델에 미니배치를 입력하여 로짓을 계산합니다.
    logits = model(input_ids, attention_mask, token_type_ids)

# 로짓 텐서의 형태를 출력합니다.
print("로짓 형태:", logits.shape)

# 로짓 일부 값을 출력합니다.
print("로짓 예시:", logits[:5])

# sigmoid를 적용하여 긍정 확률로 변환한 값을 출력합니다.
print("긍정 확률 예시:", torch.sigmoid(logits[:5]))


로짓 형태: torch.Size([16])
로짓 예시: tensor([ 0.0727, -0.5497, -0.2558,  0.2556,  0.3130], device='cuda:0')
긍정 확률 예시: tensor([0.5182, 0.3659, 0.4364, 0.5636, 0.5776], device='cuda:0')


## 모델 학습, 평가, 감성 예측

TensorFlow의 `strategy.scope()`와 TPU 설정, `model.fit()` 학습, 평가, `sentiment_predict` 함수를 이용한 긍정·부정 확률 확인을 설명합니다.

PyTorch에서는 TPU 설정 대신 CPU 또는 CUDA GPU를 자동 선택하고, `for` 반복문으로 직접 학습 루프를 작성합니다. 출력 결과 흐름에 맞추어 학습 손실, 검증 정확도, 테스트 정확도, 새 문장 예측 결과를 확인합니다.


In [20]:
# 학습률을 지정합니다. BERT 미세 조정에서는 보통 2e-5~5e-5 범위를 자주 사용합니다.
LEARNING_RATE = 5e-5

# 전체 학습 반복 횟수를 지정합니다.
EPOCHS = 2

# 이진 분류에서 로짓 입력을 직접 받는 손실함수를 생성합니다.
criterion = nn.BCEWithLogitsLoss()

# AdamW 최적화 알고리즘을 생성합니다.
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)


In [21]:
# 한 epoch 동안 모델을 학습하는 함수를 정의합니다.
def train_one_epoch(model, data_loader, criterion, optimizer, device):
    # 모델을 학습 모드로 전환하여 Dropout 등이 활성화되게 합니다.
    model.train()

    # 전체 손실 합계를 저장할 변수를 0으로 초기화합니다.
    total_loss = 0.0

    # 정답을 맞힌 샘플 수를 저장할 변수를 0으로 초기화합니다.
    total_correct = 0

    # 전체 샘플 수를 저장할 변수를 0으로 초기화합니다.
    total_count = 0

    # DataLoader에서 미니배치를 하나씩 꺼내며 반복합니다.
    for batch in tqdm(data_loader, desc="학습 중"):
        # input_ids를 학습 장치로 이동합니다.
        input_ids = batch["input_ids"].to(device)

        # attention_mask를 학습 장치로 이동합니다.
        attention_mask = batch["attention_mask"].to(device)

        # token_type_ids를 학습 장치로 이동합니다.
        token_type_ids = batch["token_type_ids"].to(device)

        # 라벨을 학습 장치로 이동합니다.
        labels = batch["label"].to(device)

        # 이전 미니배치에서 계산된 기울기를 초기화합니다.
        optimizer.zero_grad()

        # 모델에 입력을 넣어 로짓을 계산합니다.
        logits = model(input_ids, attention_mask, token_type_ids)

        # 로짓과 정답 라벨을 비교하여 손실을 계산합니다.
        loss = criterion(logits, labels)

        # 손실을 기준으로 역전파를 수행하여 기울기를 계산합니다.
        loss.backward()

        # 기울기 폭주를 막기 위해 기울기 크기를 1.0 이하로 제한합니다.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # 계산된 기울기를 사용하여 모델 파라미터를 업데이트합니다.
        optimizer.step()

        # 현재 미니배치 손실에 샘플 수를 곱하여 전체 손실 합계에 더합니다.
        total_loss += loss.item() * labels.size(0)

        # 로짓에 sigmoid를 적용한 뒤 0.5 이상이면 긍정으로 예측합니다.
        preds = (torch.sigmoid(logits) >= 0.5).float()

        # 예측값과 정답이 같은 개수를 누적합니다.
        total_correct += (preds == labels).sum().item()

        # 현재 미니배치의 샘플 수를 전체 샘플 수에 더합니다.
        total_count += labels.size(0)

    # 전체 평균 손실을 계산합니다.
    avg_loss = total_loss / total_count

    # 전체 정확도를 계산합니다.
    avg_acc = total_correct / total_count

    # 평균 손실과 정확도를 반환합니다.
    return avg_loss, avg_acc


In [22]:
# 검증 또는 테스트 단계에서 모델 성능을 평가하는 함수를 정의합니다.
def evaluate(model, data_loader, criterion, device):
    # 모델을 평가 모드로 전환하여 Dropout 등을 비활성화합니다.
    model.eval()

    # 전체 손실 합계를 저장할 변수를 0으로 초기화합니다.
    total_loss = 0.0

    # 정답을 맞힌 샘플 수를 저장할 변수를 0으로 초기화합니다.
    total_correct = 0

    # 전체 샘플 수를 저장할 변수를 0으로 초기화합니다.
    total_count = 0

    # 평가 과정에서는 기울기를 계산하지 않아 메모리 사용량과 실행 시간을 줄입니다.
    with torch.no_grad():
        # DataLoader에서 미니배치를 하나씩 꺼내며 반복합니다.
        for batch in tqdm(data_loader, desc="평가 중"):
            # input_ids를 학습 장치로 이동합니다.
            input_ids = batch["input_ids"].to(device)

            # attention_mask를 학습 장치로 이동합니다.
            attention_mask = batch["attention_mask"].to(device)

            # token_type_ids를 학습 장치로 이동합니다.
            token_type_ids = batch["token_type_ids"].to(device)

            # 라벨을 학습 장치로 이동합니다.
            labels = batch["label"].to(device)

            # 모델에 입력을 넣어 로짓을 계산합니다.
            logits = model(input_ids, attention_mask, token_type_ids)

            # 로짓과 정답 라벨을 비교하여 손실을 계산합니다.
            loss = criterion(logits, labels)

            # 현재 미니배치 손실에 샘플 수를 곱하여 전체 손실 합계에 더합니다.
            total_loss += loss.item() * labels.size(0)

            # 로짓을 확률로 바꾼 뒤 0.5 이상이면 긍정으로 예측합니다.
            preds = (torch.sigmoid(logits) >= 0.5).float()

            # 예측값과 정답이 같은 개수를 누적합니다.
            total_correct += (preds == labels).sum().item()

            # 현재 미니배치의 샘플 수를 전체 샘플 수에 더합니다.
            total_count += labels.size(0)

    # 전체 평균 손실을 계산합니다.
    avg_loss = total_loss / total_count

    # 전체 정확도를 계산합니다.
    avg_acc = total_correct / total_count

    # 평균 손실과 정확도를 반환합니다.
    return avg_loss, avg_acc


In [23]:
# 가장 좋은 검증 정확도를 저장할 변수를 0으로 초기화합니다.
best_valid_acc = 0.0

# 가장 좋은 모델 가중치를 저장할 파일 이름을 지정합니다.
best_model_path = "best_bert_nsmc_torch.pt"

# 지정한 epoch 수만큼 학습을 반복합니다.
for epoch in range(1, EPOCHS + 1):
    # 현재 epoch 번호를 출력합니다.
    print(f"\n===== Epoch {epoch}/{EPOCHS} =====")

    # 훈련 데이터로 한 epoch 학습을 수행합니다.
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)

    # 검증 데이터로 현재 모델 성능을 평가합니다.
    valid_loss, valid_acc = evaluate(model, valid_loader, criterion, device)

    # 현재 epoch의 훈련 손실과 정확도를 출력합니다.
    print(f"훈련 손실: {train_loss:.4f} | 훈련 정확도: {train_acc:.4f}")

    # 현재 epoch의 검증 손실과 정확도를 출력합니다.
    print(f"검증 손실: {valid_loss:.4f} | 검증 정확도: {valid_acc:.4f}")

    # 현재 검증 정확도가 이전 최고 정확도보다 높으면 모델을 저장합니다.
    if valid_acc > best_valid_acc:
        # 최고 검증 정확도를 현재 값으로 갱신합니다.
        best_valid_acc = valid_acc

        # 모델의 학습된 파라미터를 파일로 저장합니다.
        torch.save(model.state_dict(), best_model_path)

        # 모델이 저장되었음을 출력합니다.
        print("최고 검증 정확도 갱신, 모델 저장 완료:", best_model_path)



===== Epoch 1/2 =====


학습 중:   0%|          | 0/150 [00:00<?, ?it/s]

평가 중:   0%|          | 0/38 [00:00<?, ?it/s]

훈련 손실: 0.4331 | 훈련 정확도: 0.8054
검증 손실: 0.3566 | 검증 정확도: 0.8483
최고 검증 정확도 갱신, 모델 저장 완료: best_bert_nsmc_torch.pt

===== Epoch 2/2 =====


학습 중:   0%|          | 0/150 [00:00<?, ?it/s]

평가 중:   0%|          | 0/38 [00:00<?, ?it/s]

훈련 손실: 0.2286 | 훈련 정확도: 0.9246
검증 손실: 0.5732 | 검증 정확도: 0.8500
최고 검증 정확도 갱신, 모델 저장 완료: best_bert_nsmc_torch.pt


In [24]:
# 저장된 최고 성능 모델 가중치 파일이 존재하는지 확인합니다.
if os.path.exists(best_model_path):
    # 저장된 최고 성능 모델 가중치를 현재 모델에 불러옵니다.
    model.load_state_dict(torch.load(best_model_path, map_location=device))

    # 가중치 불러오기가 완료되었음을 출력합니다.
    print("저장된 최고 성능 모델을 불러왔습니다.")

# 테스트 데이터로 최종 모델 성능을 평가합니다.
test_loss, test_acc = evaluate(model, test_loader, criterion, device)

# 테스트 손실과 정확도를 출력합니다.
print(f"테스트 손실: {test_loss:.4f} | 테스트 정확도: {test_acc:.4f}")


저장된 최고 성능 모델을 불러왔습니다.


평가 중:   0%|          | 0/63 [00:00<?, ?it/s]

테스트 손실: 0.6284 | 테스트 정확도: 0.8410


In [25]:
# 새로운 문장의 긍정/부정 감성을 예측하는 함수를 정의합니다.
def sentiment_predict(new_sentence):
    # 모델을 평가 모드로 전환합니다.
    model.eval()

    # 입력 문장을 BERT 입력 형식으로 변환합니다.
    encoding = tokenizer(
        new_sentence,              # 예측할 새 리뷰 문장입니다.
        max_length=MAX_SEQ_LEN,     # 학습 때와 같은 최대 길이를 사용합니다.
        padding="max_length",      # 짧은 문장은 PAD 토큰으로 채웁니다.
        truncation=True,            # 긴 문장은 최대 길이에 맞게 자릅니다.
        return_tensors="pt"         # PyTorch 텐서 형태로 반환합니다.
    )

    # input_ids 텐서를 학습 장치로 이동합니다.
    input_ids = encoding["input_ids"].to(device)

    # attention_mask 텐서를 학습 장치로 이동합니다.
    attention_mask = encoding["attention_mask"].to(device)

    # token_type_ids 텐서를 학습 장치로 이동합니다.
    token_type_ids = encoding["token_type_ids"].to(device)

    # 예측 과정에서는 기울기를 계산하지 않습니다.
    with torch.no_grad():
        # 모델에 입력 문장을 넣어 로짓을 계산합니다.
        logit = model(input_ids, attention_mask, token_type_ids)

        # 로짓에 sigmoid를 적용하여 긍정 확률로 변환합니다.
        positive_score = torch.sigmoid(logit).item()

    # 긍정 확률이 0.5 이상이면 긍정 리뷰로 판단합니다.
    if positive_score >= 0.5:
        # 긍정 리뷰 확률을 퍼센트로 출력합니다.
        print(f"{positive_score * 100:.2f}% 확률로 긍정 리뷰입니다.")
    else:
        # 부정 리뷰 확률을 퍼센트로 출력합니다.
        print(f"{(1 - positive_score) * 100:.2f}% 확률로 부정 리뷰입니다.")

    # 다른 코드에서 활용할 수 있도록 긍정 확률을 반환합니다.
    return positive_score


In [26]:
# 부정에 가까운 예시 리뷰를 예측합니다.
sentiment_predict("보던거라 계속 보고 있는데 전개도 느리고 주인공도 너무 소극적으로 나와서 아쉽다")

# 긍정에 가까운 예시 리뷰를 예측합니다.
sentiment_predict("스토리는 조금 아쉬웠지만 배우들의 연기력과 음악이 정말 좋아서 끝까지 몰입해서 봤다")


99.44% 확률로 부정 리뷰입니다.
99.61% 확률로 긍정 리뷰입니다.


0.9960916638374329